**This project predicts whether a human protein is a kinase or not using sequence-derived features.**

**Dataset:**
- Human protein sequences (FASTA)
- Gene Ontology annotations

**Model:**
- Random Forest
- SMOTE for class imbalance handling

### **Install Libraries**

In [ ]:
!pip install biopython imbalanced-learn -q

### **Import Libraries**

In [ ]:
import pandas as pd
import numpy as np

from Bio import SeqIO

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from imblearn.over_sampling import SMOTE

### **Load Protein Sequences**

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith(".fasta") or file.endswith(".txt"):
            print(os.path.join(root, file))

In [19]:
from zipfile import ZipFile

with ZipFile('/content/archive.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/protein_data')

print("Extraction complete")

Extraction complete


In [20]:
records = list(
    SeqIO.parse(
        "/content/protein_data/ALL-HUMAN-0001 SEQUENCES.fasta",
        "fasta"
    )
)

print("Total proteins:", len(records))

Total proteins: 70956


### **Feature Engineering**

In [22]:
amino_acids = list("ACDEFGHIKLMNPQRSTVWY")

protein_ids = []
lengths = []

aa_features = []

for record in records:

    seq = str(record.seq)

    protein_ids.append(record.id)

    lengths.append(len(seq))

    freqs = []

    for aa in amino_acids:
        freqs.append(seq.count(aa) / len(seq))

    aa_features.append(freqs)

df = pd.DataFrame(
    aa_features,
    columns=amino_acids
)

df.insert(0, "Length", lengths)

df["Protein_ID"] = protein_ids

df.head()

,Length,A,C,D,E,F,G,H,I,K,...,N,P,Q,R,S,T,V,W,Y,Protein_ID
0,379,0.068602,0.015831,0.058047,0.068602,0.034301,0.063325,0.029024,0.068602,0.055409,...,0.034301,0.058047,0.044855,0.060686,0.047493,0.050132,0.047493,0.007916,0.047493,sp|P27361|MK03_HUMAN
1,464,0.058190,0.023707,0.062500,0.058190,0.032328,0.045259,0.025862,0.056034,0.073276,...,0.043103,0.056034,0.045259,0.038793,0.079741,0.040948,0.077586,0.008621,0.047414,sp|P53779|MK10_HUMAN
2,377,0.098143,0.029178,0.037135,0.045093,0.053050,0.045093,0.007958,0.061008,0.042440,...,0.031830,0.058355,0.029178,0.042440,0.100796,0.042440,0.087533,0.013263,0.021220,sp|Q15049|MLC1_HUMAN
3,1453,0.047488,0.029594,0.048176,0.083276,0.044735,0.046800,0.018582,0.039229,0.072953,...,0.053682,0.039917,0.053682,0.051617,0.104611,0.063317,0.058500,0.005506,0.019270,sp|Q9UHC1|MLH3_HUMAN
4,46,0.000000,0.000000,0.043478,0.043478,0.043478,0.043478,0.000000,0.195652,0.065217,...,0.021739,0.021739,0.000000,0.021739,0.086957,0.108696,0.108696,0.021739,0.021739,sp|P0DMT0|MLN_HUMAN


### **Create Kinase Labels**

In [23]:
with open(
    "/content/protein_data/ALL-HUMAN-0001-ANNOTATIONS.txt"
) as f:

    lines = f.readlines()

kinase_proteins = set()

for line in lines:

    if "kinase activity" in line.lower():

        protein_id = line.split()[0]

        kinase_proteins.add(protein_id)

df["Accession"] = df["Protein_ID"].apply(
    lambda x: x.split("|")[1]
)

df["Kinase"] = (
    df["Accession"]
    .isin(kinase_proteins)
    .astype(int)
)

print(df["Kinase"].value_counts())

Kinase
0    69079
1     1877
Name: count, dtype: int64


### **Train Test Split**

In [24]:
X = df.drop(
    columns=[
        "Protein_ID",
        "Accession",
        "Kinase"
    ]
)

y = df["Kinase"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(56764, 21)
(14192, 21)


### **Apply SMOTE**

In [25]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print(y_train_smote.value_counts())

Kinase
0    55262
1    55262
Name: count, dtype: int64


### **Train Random Forest**

In [26]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    X_train_smote,
    y_train_smote
)

preds = rf.predict(X_test)

### **Evaluation**

In [27]:
print(
    "Accuracy:",
    accuracy_score(y_test, preds)
)

print(
    "Precision:",
    precision_score(y_test, preds)
)

print(
    "Recall:",
    recall_score(y_test, preds)
)

print(
    "F1 Score:",
    f1_score(y_test, preds)
)

print(
    classification_report(
        y_test,
        preds
    )
)

Accuracy: 0.9553269447576099
Precision: 0.2763385146804836
Recall: 0.4266666666666667
F1 Score: 0.33542976939203356
              precision    recall  f1-score   support

           0       0.98      0.97      0.98     13817
           1       0.28      0.43      0.34       375

    accuracy                           0.96     14192
   macro avg       0.63      0.70      0.66     14192
weighted avg       0.97      0.96      0.96     14192



### **Feature Importance**

In [28]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance.head(15)

,Feature,Importance
4,E,0.083294
8,I,0.072457
5,F,0.070240
7,H,0.063888
0,Length,0.059323
6,G,0.055038
9,K,0.053561
10,L,0.050274
3,D,0.049252
15,R,0.047099


### **Save Feature Importance**

In [29]:
importance.to_csv(
    "feature_importance.csv",
    index=False
)

print("Saved successfully")

Saved successfully
